In [1]:
import pandas as pd

df_train = pd.read_parquet("../data/processed/train_clean.parquet")
df_test = pd.read_parquet("../data/processed/test_clean.parquet")

In [2]:
df_train.shape

(25000, 3)

In [3]:
import torch
from collections import Counter

In [6]:
all_words = []

for t in df_train.text:
    all_words.extend(t.split())

5719421

In [7]:
all_words_counts = Counter(all_words)
all_words_counts.most_common(5)

[('the', 328018),
 ('and', 161560),
 ('a', 161291),
 ('of', 145156),
 ('to', 134816)]

In [8]:
word_freq = {word: count for word, count in all_words_counts.items() if count >= 2}
len(word_freq)

57232

In [9]:
word2dix = {"<pad>":0, "<unk>" : 1}

for i, word in enumerate(word_freq.keys()):
    word2dix[word] = i + 2

len(word2dix)


57234

In [10]:
word2dix['the']

13

In [12]:
def text_to_indices(text, word2idx, max_len=512):
    words_list = text.split()

    if len(words_list) > max_len:
        words_list = words_list[:max_len]

    unk_index = word2idx.get("<unk>", 0)

    indices = [word2idx.get(item, unk_index) for item in words_list]
    return indices

In [13]:
text_to_indices(df_train["text"][0], word2dix)

[2,
 3,
 2,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 17,
 19,
 20,
 21,
 22,
 23,
 2,
 24,
 25,
 15,
 26,
 20,
 17,
 19,
 27,
 28,
 29,
 30,
 31,
 17,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 11,
 42,
 43,
 44,
 2,
 45,
 46,
 34,
 47,
 36,
 48,
 49,
 50,
 51,
 52,
 53,
 40,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 34,
 62,
 63,
 64,
 65,
 66,
 67,
 22,
 68,
 64,
 61,
 34,
 69,
 70,
 71,
 34,
 72,
 73,
 74,
 11,
 75,
 76,
 77,
 13,
 78,
 79,
 80,
 66,
 81,
 82,
 83,
 84,
 85,
 13,
 86,
 87,
 88,
 89,
 83,
 22,
 13,
 90,
 91,
 22,
 92,
 93,
 94,
 88,
 95,
 96,
 11,
 97,
 66,
 98,
 99,
 76,
 100,
 64,
 101,
 102,
 103,
 70,
 56,
 104,
 105,
 88,
 106,
 1,
 107,
 108,
 66,
 2,
 4,
 5,
 51,
 15,
 109,
 110,
 111,
 36,
 19,
 43,
 112,
 45,
 13,
 102,
 88,
 113,
 114,
 115,
 116,
 88,
 117,
 92,
 118,
 119,
 120,
 121,
 122,
 123,
 73,
 124,
 125,
 126,
 127,
 7,
 128,
 129,
 130,
 17,
 131,
 22,
 132,
 102,
 88,
 113,
 115,
 40,
 133,
 134,
 22,
 55,
 1

In [15]:
from torch.utils.data import Dataset
class IMDbDataset(Dataset):

    def __init__(self, texts, labels, word2dix):
        super().__init__()
        self.texts = texts
        self.labels = labels
        self.word2dix = word2dix


    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = text_to_indices(self.texts[idx], self.word2dix)
        target = self.labels[idx]
        return text, target


In [18]:
dataset = IMDbDataset(df_train["text"].values, df_train["label"].values, word2dix)

text, label = dataset[99]

In [19]:
print(text, label)

[36, 153, 51, 1585, 267, 211, 45, 1753, 34, 625, 36, 1744, 1860, 31, 267, 115, 4253, 76, 622, 17, 4254, 34, 370, 211, 316, 267, 115, 2169, 551, 121, 34, 161, 40, 140, 3670, 342, 51, 574, 1646, 503, 2011, 1447, 13, 374, 2, 80, 15, 17, 19, 293, 15, 203, 1028, 582, 60, 4255, 88, 587, 123, 2062, 4256, 204, 34, 130, 328, 15, 17, 51, 1412, 678, 40, 314, 1041, 118, 48, 4257, 50, 51, 1585, 160, 60, 191, 15, 17, 51, 1336, 241, 140, 176, 332, 523, 200, 40, 1809, 374, 239, 118, 28, 13, 280, 11, 771, 1717, 4258, 36, 51, 40, 1585, 4259, 800, 17, 540, 48, 40, 4260, 995, 26, 584, 267, 269, 299, 123, 968, 1589] 0


In [20]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    texts, labels = zip(*batch)
    collated = pad_sequence([torch.tensor(t) for t in texts], batch_first=True, padding_value=0)
    return collated, torch.tensor(labels)



In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

texts, labels = next(iter(train_loader))
print(texts.shape, labels.shape)

torch.Size([64, 512]) torch.Size([64])


In [48]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.3, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits

In [49]:
model = LSTMClassifier(
    vocab_size=len(word2dix),
    embedding_dim=128,
    hidden_size=256,
    num_layers=2
)
print(model)

LSTMClassifier(
  (embedding): Embedding(57234, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [50]:
if torch.backends.mps.is_available():
    device = torch.device("mps")

model = model.to(device)
print(next(model.parameters()).device)

mps:0


In [51]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
epochs = 5
losses = []

for i in range(epochs):
    epoch_loss = 0
    for text, label in train_loader:
        text = text.to(device)
        label = label.to(device).float()

        y_pred = model(text)
        print(y_pred.shape)
        break
        loss = criterion(y_pred.squeeze(1), label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f"Epoch {i} loss is : {epoch_loss / len(train_loader)}")

In [53]:
epochs = 3
losses = []

for i in range(epochs):
    epoch_loss = 0
    for text, label in train_loader:
        text = text.to(device)
        label = label.to(device).float()

        y_pred = model(text)

        loss = criterion(y_pred.squeeze(1), label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f"Epoch {i} loss is : {epoch_loss / len(train_loader)}")

Epoch 0 loss is : 0.6694547160507163
Epoch 1 loss is : 0.6012067097379729
Epoch 2 loss is : 0.5897380260707777


In [54]:
epochs = 10
losses = []

for i in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for text, label in train_loader:
        text = text.to(device)
        label = label.to(device).float()

        y_pred = model(text)
        loss = criterion(y_pred.squeeze(1), label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        predicted = (torch.sigmoid(y_pred.squeeze(1)) >= 0.5).float()
        correct += (predicted == label).sum().item()
        total += label.size(0)

    acc = correct / total * 100
    print(f"Epoch {i} | Loss: {epoch_loss / len(train_loader):.4f} | Accuracy: {acc:.2f}%")

Epoch 0 | Loss: 0.6011 | Accuracy: 67.60%
Epoch 1 | Loss: 0.5178 | Accuracy: 75.04%
Epoch 2 | Loss: 0.4700 | Accuracy: 78.42%
Epoch 3 | Loss: 0.4595 | Accuracy: 79.45%
Epoch 4 | Loss: 0.5850 | Accuracy: 68.54%
Epoch 5 | Loss: 0.6468 | Accuracy: 62.22%
Epoch 6 | Loss: 0.5518 | Accuracy: 72.64%
Epoch 7 | Loss: 0.4575 | Accuracy: 78.95%
Epoch 8 | Loss: 0.3945 | Accuracy: 82.74%
Epoch 9 | Loss: 0.3594 | Accuracy: 84.91%


In [55]:
df_test.head()

,text,label
0,i love scifi and am willing to put up with a l...,0
1,worth the entertainment value of a rental espe...,0
2,its a totally average film with a few semialri...,0
3,star rating saturday night friday night fri...,0
4,first off let me say if you havent enjoyed a v...,0


In [59]:
test_dataset = IMDbDataset(df_test["text"].values, df_test["label"].values, word2dix)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
text, label = next(iter(test_loader))

In [61]:
correct = 0
total = 0

with torch.no_grad():
    for text_batch, label_batch in test_loader:
        text_batch = text_batch.to(device)
        label_batch = label_batch.to(device).float()

        y_val = model(text_batch).squeeze(1)
        preds = (y_val > 0).float()

        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")
print(f"Correctly predicted {correct} out of {total} samples")

Test Accuracy: 81.87%
Correctly predicted 20468 out of 25000 samples


In [62]:
torch.save(model.state_dict(), '../models/lstm_model.pt')